# Phase 1 — Load, clean, and **audit** the data

**Why audit before modelling?** (paper §3.3) A model is only as trustworthy as the
data underneath it. Before training anything we check three things:

1. **Duplicates / redundancy** — street photos taken seconds apart look identical. If
   near-duplicates end up on both sides of a train/test split, the model can "cheat"
   by recognising a scene it already saw. We measure how much redundancy exists.
2. **Images per station** — decides whether our leakage-safe split (grouping by
   station) is feasible.
3. **A physics sanity check** — hazier photos should have lower "transmission" and
   higher AQI. If that relationship is missing, the images and labels were mis-paired,
   and nothing downstream would be meaningful.

We also **clean** the data: drop dead columns, remove duplicate rows, and shuffle.

## Bootstrap (connect Drive + make `src` importable)

In [ ]:
import os, sys
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    if os.path.isdir("/content/pm25-visual-aq"):
        os.chdir("/content/pm25-visual-aq")
except ImportError:
    pass  # running locally, not on Colab
sys.path.insert(0, os.getcwd())

from src.config import load_config
from src import data, audit
cfg = load_config()

# Full dataset (Colab). For a quick laptop test, set SOURCE = "tests/fixture_ds".
SOURCE = cfg["data"]["drive_path"]
print("Loading from:", SOURCE)

## Load + clean

`load_clean` pools the train/test splits into one pool (we make our own splits in
Phase 2), drops columns that carry no signal, removes duplicate `image_id` rows, and
shuffles with a fixed seed so file ordering can never bias a split.

In [ ]:
ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
print("rows after cleaning:", len(df))
print("duplicate rows removed:", df.attrs.get("n_duplicates_removed"))
df.head()

## The label: it's an **AQI index**, not µg/m³

PM25Vision's label is a US-EPA Air Quality Index value (roughly 1–530), a unitless
index — *not* a raw concentration. Every error we report later is in **AQI points**.
The histogram is right-skewed (many moderate days, few extreme ones), which is why we
later train on `log(AQI)`.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.hist(df["pm25"], bins=50)
plt.xlabel("pm25 (AQI index)"); plt.ylabel("number of photos")
plt.title("Label distribution — right-skewed"); plt.show()
print(df["pm25"].describe())

## Images per station

This decides whether we can split *by station* (our leakage-safe protocol). With
thousands of stations and most contributing only a couple of images, grouping is
easy. The few stations with many images are where near-duplicate risk concentrates.

In [ ]:
sps = audit.images_per_station(df, station_col=cfg["data"]["station_col"])
print("stations: %d | images/station  median=%.1f  mean=%.2f  max=%d  (%.0f%% have just 1)"
      % (sps["n_stations"], sps["median"], sps["mean"], sps["max"], sps["pct_single_image"]))
plt.figure(figsize=(7,3))
plt.hist(sps["counts"].values, bins=40)
plt.xlabel("images at one station"); plt.ylabel("number of stations")
plt.title("Most stations contribute only a few images"); plt.show()

## Near-duplicate check (perceptual hashing)

A *perceptual hash* is a short fingerprint where **similar-looking images get similar
fingerprints**. We fingerprint every image and group ones whose fingerprints differ
by only a few bits. The **distinct-ratio** = groups ÷ images: 1.0 means no
near-duplicates; lower means redundancy we must keep out of the split.

> ⏳ On the full dataset this decodes ~11k images and takes a few minutes.

In [ ]:
rep = audit.redundancy_report(ds, df, hash_size=8, max_distance=5)
print("images: %d | distinct groups: %d | distinct-ratio: %.3f | largest group: %d"
      % (rep["n_images"], rep["n_distinct_groups"], rep["distinct_ratio"], rep["largest_group"]))

## Physics falsification test

Physics says hazier photos have **lower transmission** and **higher AQI**, so average
transmission should be **negatively** correlated with the label. A clearly negative
correlation means images and labels line up — the learning problem is real.

In [ ]:
cor = audit.transmission_label_correlation(
    ds, df, target_col=cfg["data"]["target_col"], sample=1000, seed=cfg["seed"])
print("Pearson r = %.3f (p=%.1e)  |  Spearman r = %.3f  |  negative as expected? %s"
      % (cor["pearson_r"], cor["pearson_p"], cor["spearman_r"], cor["passes"]))
plt.figure(figsize=(5,4))
plt.scatter(cor["mean_transmission"], cor["labels"], s=6, alpha=0.4)
plt.xlabel("average transmission (clearer →)"); plt.ylabel("AQI label")
plt.title("Should slope downward"); plt.show()

## Where in the world are these photos?

Coverage is heavily skewed toward East Asia, Europe, and India, with little in the
Americas or Africa. That's fine, but it means any "generalises everywhere" claim must
be scoped to the regions actually represented (paper §3.2).

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(df[cfg["data"]["lon_col"]], df[cfg["data"]["lat_col"]], s=4, alpha=0.3)
plt.xlabel("longitude"); plt.ylabel("latitude"); plt.title("Station geography"); plt.show()

## What we found (summary)

- Label is **AQI (1–530)**, right-skewed → we'll train on `log(AQI)`.
- **3,261 stations**, most with only a few images → station-grouped split is easy.
- Near-duplicate redundancy is concentrated in a handful of busy stations.
- The physics check should be **negative** — confirming images and labels match.
- Geography is skewed → we scope our claims honestly.

**Next:** `notebooks/02_splits.ipynb` — building leakage-safe train/calibration/test
splits, and measuring how much a naive random split inflates results.